In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

ROOT = r"C:\Users\ROHITH KANNA S\WiFiVision"

CSI_DIR = os.path.join(
    ROOT,
    "data",
    "processed",
    "mmfi_calibrated"
)

SEQ_DIR = os.path.join(
    ROOT,
    "data",
    "processed",
    "mmfi_sequence"
)

SPLIT_DIR = os.path.join(
    ROOT,
    "data",
    "processed",
    "baseline_split"
)

print("CSI directory:", CSI_DIR)
print("Sequence directory:", SEQ_DIR)
print("Split directory:", SPLIT_DIR)

CSI directory: C:\Users\ROHITH KANNA S\WiFiVision\data\processed\mmfi_calibrated
Sequence directory: C:\Users\ROHITH KANNA S\WiFiVision\data\processed\mmfi_sequence
Split directory: C:\Users\ROHITH KANNA S\WiFiVision\data\processed\baseline_split


In [4]:
# ============================================================
# LOAD CALIBRATED MM-Fi DATA
# ============================================================

import os
import numpy as np

ROOT = r"C:\Users\ROHITH KANNA S\WiFiVision"

CSI_DIR = os.path.join(
    ROOT,
    "data",
    "processed",
    "mmfi_calibrated"
)

amplitude = np.load(
    os.path.join(CSI_DIR, "amplitude_clean.npy"),
    mmap_mode="r"
)

phase = np.load(
    os.path.join(CSI_DIR, "phase_calibrated.npy"),
    mmap_mode="r"
)

pose = np.load(
    os.path.join(CSI_DIR, "pose.npy"),
    mmap_mode="r"
)

subjects = np.load(
    os.path.join(CSI_DIR, "subject.npy")
)

actions = np.load(
    os.path.join(CSI_DIR, "action.npy")
)

print("Loaded successfully.")
print()
print("Amplitude:", amplitude.shape, amplitude.dtype)
print("Phase    :", phase.shape, phase.dtype)
print("Pose     :", pose.shape, pose.dtype)
print("Subjects :", subjects.shape, np.unique(subjects))
print("Actions  :", actions.shape, np.unique(actions))

Loaded successfully.

Amplitude: (270, 297, 3, 114, 10) float32
Phase    : (270, 297, 3, 114, 10) float32
Pose     : (270, 297, 17, 2) float32
Subjects : (270,) [ 1  2  3  4  5  6  7  8  9 10]
Actions  : (270,) [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27]


In [5]:
# ============================================================
# PHASE-ENHANCED DATASET
# ============================================================

from torch.utils.data import Dataset

TRAIN_SUBJECTS = [1, 2, 3, 4, 5, 6, 7]
VAL_SUBJECTS = [8]
TEST_SUBJECTS = [9, 10]

WINDOW = 30
STRIDE = 15

train_seq = np.where(
    np.isin(subjects, TRAIN_SUBJECTS)
)[0]

val_seq = np.where(
    np.isin(subjects, VAL_SUBJECTS)
)[0]

test_seq = np.where(
    np.isin(subjects, TEST_SUBJECTS)
)[0]


# Training-only pose normalization
pose_mean = np.asarray(
    pose[train_seq].mean(axis=(0, 1)),
    dtype=np.float32
)

pose_std = np.asarray(
    pose[train_seq].std(axis=(0, 1)),
    dtype=np.float32
)

pose_std = np.maximum(
    pose_std,
    1e-6
)


class PhaseEnhancedDataset(Dataset):

    def __init__(
        self,
        sequence_indices
    ):

        self.sequence_indices = sequence_indices

        self.windows = []

        for seq in sequence_indices:

            for start in range(
                0,
                amplitude.shape[1] - WINDOW + 1,
                STRIDE
            ):

                self.windows.append(
                    (
                        int(seq),
                        int(start)
                    )
                )

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):

        seq, start = self.windows[index]

        end = start + WINDOW

        # Amplitude
        amp = np.asarray(
            amplitude[
                seq,
                start:end
            ],
            dtype=np.float32
        )

        # Calibrated phase
        ph = np.asarray(
            phase[
                seq,
                start:end
            ],
            dtype=np.float32
        )

        # Circular phase representation
        sin_ph = np.sin(ph)
        cos_ph = np.cos(ph)

        # ----------------------------------------------------
        # 3 amplitude channels
        # + 3 sin(phase) channels
        # + 3 cos(phase) channels
        #
        # = 9 CSI channels
        # ----------------------------------------------------

        x = np.concatenate(
            [
                amp,
                sin_ph,
                cos_ph
            ],
            axis=1
        )

        # Central/target frame
        target_frame = end - 1

        y = np.asarray(
            pose[
                seq,
                target_frame
            ],
            dtype=np.float32
        )

        y = (
            y - pose_mean
        ) / pose_std

        return (
            torch.from_numpy(x.copy()),
            torch.from_numpy(y.copy())
        )


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = PhaseEnhancedDataset(
    train_seq
)

val_dataset = PhaseEnhancedDataset(
    val_seq
)

test_dataset = PhaseEnhancedDataset(
    test_seq
)


print("Train sequences:", len(train_seq))
print("Validation sequences:", len(val_seq))
print("Test sequences:", len(test_seq))

print()

print("Train windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
print("Test windows:", len(test_dataset))


# ============================================================
# CHECK ONE SAMPLE
# ============================================================

x, y = train_dataset[0]

print()
print("Sample CSI:", tuple(x.shape))
print("Sample pose:", tuple(y.shape))
print("CSI dtype:", x.dtype)
print("Pose dtype:", y.dtype)

Train sequences: 189
Validation sequences: 27
Test sequences: 54

Train windows: 3402
Validation windows: 486
Test windows: 972

Sample CSI: (30, 9, 114, 10)
Sample pose: (17, 2)
CSI dtype: torch.float32
Pose dtype: torch.float32


In [6]:
import torch
import torch.nn as nn


class PhaseEnhancedFrameCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=9,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):

        x = self.features(x)

        return x.flatten(1)


class CSIPhase2Pose(nn.Module):

    def __init__(self):
        super().__init__()

        self.frame_cnn = PhaseEnhancedFrameCNN()

        self.gru = nn.GRU(
            input_size=32,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(

            nn.Linear(64, 128),

            nn.ReLU(),

            nn.Linear(128, 34)
        )

    def forward(self, x):

        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        # (B, 30, 9, 114, 10)
        # →
        # (B*30, 9, 114, 10)

        x = x.reshape(
            batch_size * sequence_length,
            9,
            114,
            10
        )

        x = self.frame_cnn(x)

        # (B*30, 32)
        # →
        # (B, 30, 32)

        x = x.reshape(
            batch_size,
            sequence_length,
            32
        )

        x, _ = self.gru(x)

        # Final temporal representation
        x = x[:, -1, :]

        x = self.head(x)

        return x.reshape(
            batch_size,
            17,
            2
        )


# ============================================================
# MODEL CHECK
# ============================================================

model = CSIPhase2Pose()

dummy = torch.randn(
    4,
    30,
    9,
    114,
    10
)

output = model(dummy)

print("Input :", tuple(dummy.shape))
print("Output:", tuple(output.shape))

parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", parameters)

Input : (4, 30, 9, 114, 10)
Output: (4, 17, 2)
Parameters: 37570


In [7]:
# ============================================================
# TRAIN PHASE-ENHANCED MODEL
# ============================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 5

DEVICE = torch.device("cpu")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

model = CSIPhase2Pose().to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

best_val_loss = float("inf")
patience_counter = 0

os.makedirs(
    os.path.join(ROOT, "models"),
    exist_ok=True
)

checkpoint_path = os.path.join(
    ROOT,
    "models",
    "phase_best.pt"
)

print("========================================")
print("PHASE-ENHANCED TRAINING")
print("========================================")

for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0

    for x, y in train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        prediction = model(x)

        loss = criterion(
            prediction,
            y
        )

        loss.backward()
        optimizer.step()

        train_loss += (
            loss.item() * x.size(0)
        )

    train_loss /= len(train_dataset)


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            prediction = model(x)

            loss = criterion(
                prediction,
                y
            )

            val_loss += (
                loss.item() * x.size(0)
            )

    val_loss /= len(val_dataset)


    # --------------------------------------------------------
    # REPORT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "val_loss":
                    val_loss
            },
            checkpoint_path
        )

        print("  -> Best model saved")

    else:

        patience_counter += 1

        print(
            f"  -> No improvement "
            f"({patience_counter}/{PATIENCE})"
        )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if patience_counter >= PATIENCE:

        print("\nEarly stopping.")
        break


print("\n========================================")
print("TRAINING COMPLETE")
print("========================================")

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Checkpoint:",
    checkpoint_path
)

PHASE-ENHANCED TRAINING
Epoch 01/30 | Train Loss: 0.988264 | Val Loss: 1.215697
  -> Best model saved
Epoch 02/30 | Train Loss: 0.945657 | Val Loss: 1.321314
  -> No improvement (1/5)
Epoch 03/30 | Train Loss: 0.922020 | Val Loss: 1.302829
  -> No improvement (2/5)
Epoch 04/30 | Train Loss: 0.893526 | Val Loss: 1.094151
  -> Best model saved
Epoch 05/30 | Train Loss: 0.863380 | Val Loss: 2.257301
  -> No improvement (1/5)
Epoch 06/30 | Train Loss: 0.826002 | Val Loss: 1.187541
  -> No improvement (2/5)
Epoch 07/30 | Train Loss: 0.807621 | Val Loss: 1.199665
  -> No improvement (3/5)
Epoch 08/30 | Train Loss: 0.773621 | Val Loss: 1.119877
  -> No improvement (4/5)
Epoch 09/30 | Train Loss: 0.747127 | Val Loss: 1.426988
  -> No improvement (5/5)

Early stopping.

TRAINING COMPLETE
Best validation loss: 1.0941513987725655
Checkpoint: C:\Users\ROHITH KANNA S\WiFiVision\models\phase_best.pt


In [8]:
# ============================================================
# EVALUATE PHASE-ENHANCED MODEL
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

# Load best validation checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(DEVICE)

        prediction = model(x)

        all_predictions.append(
            prediction.cpu().numpy()
        )

        all_targets.append(
            y.numpy()
        )

pred_phase = np.concatenate(
    all_predictions,
    axis=0
)

target_phase = np.concatenate(
    all_targets,
    axis=0
)

# ------------------------------------------------------------
# Convert normalized coordinates back
# ------------------------------------------------------------

pred_phase = (
    pred_phase * pose_std
    + pose_mean
)

target_phase = (
    target_phase * pose_std
    + pose_mean
)

# ------------------------------------------------------------
# Joint Euclidean error
# ------------------------------------------------------------

phase_errors = np.sqrt(
    np.sum(
        (pred_phase - target_phase) ** 2,
        axis=2
    )
)

phase_mpjpe = np.mean(
    phase_errors
)

# ------------------------------------------------------------
# PCK
# ------------------------------------------------------------

thresholds = [0.05, 0.10, 0.15, 0.20]

phase_pck = {}

for threshold in thresholds:

    phase_pck[threshold] = np.mean(
        phase_errors <= threshold
    )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("========================================")
print("PHASE-ENHANCED TEST EVALUATION")
print("========================================")

print("Test subjects:", TEST_SUBJECTS)
print("Test windows :", len(test_dataset))

print()

print(
    f"MPJPE: {phase_mpjpe:.6f}"
)

print()

print("PCK:")

for threshold in thresholds:

    print(
        f"@ {threshold:.2f}: "
        f"{phase_pck[threshold] * 100:.2f}%"
    )

PHASE-ENHANCED TEST EVALUATION
Test subjects: [9, 10]
Test windows : 972

MPJPE: 0.146023

PCK:
@ 0.05: 17.97%
@ 0.10: 50.92%
@ 0.15: 69.17%
@ 0.20: 77.30%


In [9]:
# ============================================================
# EXPERIMENT C — RICHER SPATIAL CNN
# ============================================================

class SpatialFrameCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # ------------------------------------------------
            # Block 1
            # ------------------------------------------------

            nn.Conv2d(
                3, 16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # ------------------------------------------------
            # Block 2
            # ------------------------------------------------

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # ------------------------------------------------
            # Block 3
            # ------------------------------------------------

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),
            nn.ReLU(),

            # ------------------------------------------------
            # Preserve spatial structure longer
            # ------------------------------------------------

            nn.AdaptiveAvgPool2d(
                (4, 2)
            )
        )

        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 4 * 2,
                64
            ),

            nn.ReLU()
        )


    def forward(self, x):

        x = self.features(x)

        x = self.projection(x)

        return x


class SpatialCSI2Pose(nn.Module):

    def __init__(self):
        super().__init__()

        self.frame_cnn = SpatialFrameCNN()

        self.gru = nn.GRU(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(

            nn.Linear(
                64,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                34
            )
        )


    def forward(self, x):

        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        # (B, 30, 3, 114, 10)
        x = x.reshape(
            batch_size * sequence_length,
            3,
            114,
            10
        )

        x = self.frame_cnn(x)

        # (B*30, 64)
        x = x.reshape(
            batch_size,
            sequence_length,
            64
        )

        x, _ = self.gru(x)

        # Final temporal representation
        x = x[:, -1, :]

        x = self.head(x)

        return x.reshape(
            batch_size,
            17,
            2
        )


# ============================================================
# CHECK
# ============================================================

spatial_model = SpatialCSI2Pose()

dummy = torch.randn(
    4,
    30,
    3,
    114,
    10
)

output = spatial_model(dummy)

parameters = sum(
    p.numel()
    for p in spatial_model.parameters()
)

print("Input :", tuple(dummy.shape))
print("Output:", tuple(output.shape))
print("Parameters:", parameters)

Input : (4, 30, 3, 114, 10)
Output: (4, 17, 2)
Parameters: 94306


In [ ]:
# ============================================================
# EXPERIMENT C — AMPLITUDE-ONLY SPATIAL DATASET + TRAINING
# ============================================================

from torch.utils.data import Dataset, DataLoader


# ============================================================
# AMPLITUDE-ONLY DATASET
# ============================================================

class AmplitudeDataset(Dataset):

    def __init__(
        self,
        sequence_indices
    ):

        self.sequence_indices = sequence_indices
        self.windows = []

        for seq in sequence_indices:

            for start in range(
                0,
                amplitude.shape[1] - WINDOW + 1,
                STRIDE
            ):

                self.windows.append(
                    (
                        int(seq),
                        int(start)
                    )
                )

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):

        seq, start = self.windows[index]

        end = start + WINDOW

        # ----------------------------------------------------
        # AMPLITUDE ONLY
        #
        # (30, 3, 114, 10)
        # ----------------------------------------------------

        x = np.asarray(
            amplitude[
                seq,
                start:end
            ],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # TARGET FRAME
        # ----------------------------------------------------

        target_frame = end - 1

        y = np.asarray(
            pose[
                seq,
                target_frame
            ],
            dtype=np.float32
        )

        # Training-only normalization
        y = (
            y - pose_mean
        ) / pose_std

        return (
            torch.from_numpy(x.copy()),
            torch.from_numpy(y.copy())
        )


# ============================================================
# CREATE CORRECT DATASETS
# ============================================================

amp_train_dataset = AmplitudeDataset(
    train_seq
)

amp_val_dataset = AmplitudeDataset(
    val_seq
)

amp_test_dataset = AmplitudeDataset(
    test_seq
)


print("========================================")
print("EXPERIMENT C DATASET")
print("========================================")

print(
    "Train windows:",
    len(amp_train_dataset)
)

print(
    "Validation windows:",
    len(amp_val_dataset)
)

print(
    "Test windows:",
    len(amp_test_dataset)
)


# ============================================================
# VERIFY INPUT SHAPE BEFORE TRAINING
# ============================================================

sample_x, sample_y = amp_train_dataset[0]

print()
print(
    "Sample CSI shape:",
    tuple(sample_x.shape)
)

print(
    "Sample pose shape:",
    tuple(sample_y.shape)
)


# ============================================================
# DATALOADERS
# ============================================================

amp_train_loader = DataLoader(
    amp_train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

amp_val_loader = DataLoader(
    amp_val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


# ============================================================
# MODEL
# ============================================================

DEVICE = torch.device("cpu")

spatial_model = SpatialCSI2Pose().to(
    DEVICE
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    spatial_model.parameters(),
    lr=1e-3
)

EPOCHS = 30
PATIENCE = 5

best_val_loss = float("inf")
patience_counter = 0

spatial_checkpoint = os.path.join(
    ROOT,
    "models",
    "spatial_best.pt"
)


# ============================================================
# TRAIN
# ============================================================

print()
print("========================================")
print("EXPERIMENT C — RICHER SPATIAL CNN")
print("========================================")

for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    spatial_model.train()

    train_loss = 0.0

    for x, y in amp_train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        prediction = spatial_model(x)

        loss = criterion(
            prediction,
            y
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() * x.size(0)
        )

    train_loss /= len(
        amp_train_dataset
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    spatial_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for x, y in amp_val_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            prediction = spatial_model(x)

            loss = criterion(
                prediction,
                y
            )

            val_loss += (
                loss.item() * x.size(0)
            )

    val_loss /= len(
        amp_val_dataset
    )


    # --------------------------------------------------------
    # REPORT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    spatial_model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "val_loss":
                    val_loss
            },
            spatial_checkpoint
        )

        print("  -> Best model saved")

    else:

        patience_counter += 1

        print(
            f"  -> No improvement "
            f"({patience_counter}/{PATIENCE})"
        )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if patience_counter >= PATIENCE:

        print()
        print("Early stopping.")
        break


print()
print("========================================")
print("EXPERIMENT C COMPLETE")
print("========================================")

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Checkpoint:",
    spatial_checkpoint
)

EXPERIMENT C DATASET
Train windows: 3402
Validation windows: 486
Test windows: 972

Sample CSI shape: (30, 3, 114, 10)
Sample pose shape: (17, 2)

EXPERIMENT C — RICHER SPATIAL CNN
Epoch 01/30 | Train Loss: 0.972847 | Val Loss: 1.474588
  -> Best model saved
Epoch 02/30 | Train Loss: 0.911154 | Val Loss: 1.242667
  -> Best model saved
